# Séance 2 : production solaire, prix de l'électricité et CO₂

### Imports

In [ ]:
# À exécuter en premier sur Google Colab (sans effet en local) : récupère les données du cours
import os, sys
if "google.colab" in sys.modules:
    DEPOT = "https://github.com/borges67/export_cours_modelisation.git"
    DOSSIER = DEPOT.rstrip("/").split("/")[-1].removesuffix(".git")
    if not os.path.exists(f"/content/{DOSSIER}"):
        !git clone -q --depth 1 {DEPOT} /content/{DOSSIER}
    os.chdir(f"/content/{DOSSIER}")
print(os.getcwd())

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

pd.set_option("display.width", 140)
plt.rcParams["figure.figsize"] = (10, 3.5)

BATIMENTS = ["bureau_de_poste", "bureaux", "plateforme_tri", "plateforme_distribution"]

## 1. Construction de la courbe de production

Règles à prednre en compte :

| Obligation | Exigence |
|---|---|
| parkings extérieurs existants | ombrières photovoltaïques sur au moins la moitié de la surface |
| toitures neuves ou rénovées | 30 %, puis 40 % (2026), puis 50 % (2027) de la toiture en photovoltaïque ou végétalisée |

In [ ]:
type_bat = ...

# Type : bureaux (données LPI, onglet batiment)
SURFACE_PARKING =                           # Surface totale du parking (m²)
EMPRISE_BAT =                               # Surface totale de la toiture du bâtiment (m²)

# Calculez les surfaces de panneaux imposées par les obligations (tableau ci-dessus)
ombriere_pv_m2 = ...                            # Surface PV sur parking
toiture_pv_m2 = ...                             # Surface PV sur toiture
total_pv_m2 = ...   # Surface PV totale

**Facteur de charge** horaire du solaire pour les quatre années climatiques : `data/interne/courbe_pv.csv`

In [ ]:
PV = pd.read_csv("data/interne/courbe_pv.csv", index_col="date_heure", parse_dates=True)  # facteur de charge (kW produits par kW installé)

PV.sum().round(0)  # heures équivalentes à pleine puissance (kWh par kWc et par an)

In [ ]:
juin, decembre = PV.index.month == 6, PV.index.month == 12
fig, axes = plt.subplots(1, 3, figsize=(16, 4))
for annee in PV.columns:
    axes[0].plot(PV.loc[juin, annee].groupby(PV.index.hour[juin]).mean(), label=annee)
    axes[1].plot(PV.loc[decembre, annee].groupby(PV.index.hour[decembre]).mean(), label=annee)
    axes[2].plot(range(1, 13), PV[annee].groupby(PV.index.month).sum(), marker="o", label=annee)
axes[0].set_title("journée moyenne de juin")
axes[1].set_title("journée moyenne de décembre")
axes[2].set_title("production mensuelle (kWh par kWc)")
axes[0].set_ylabel("facteur de charge")
axes[0].set_xlabel("heure (UTC+1)")
axes[0].set_ylim(0, 0.7)
axes[1].set_ylim(0, 0.7)
axes[2].set_ylim(0, 160)
axes[1].set_xlabel("heure (UTC+1)")
axes[2].set_xlabel("mois")
axes[2].legend()
for ax in axes:
    ax.grid(alpha=0.3)
fig.tight_layout()
plt.show()

## 2. Production et consommation des sites

La consommation d'un site de chaque type vient de la séance 1 :
`data/resultats/consommation/<horizon>/<bâtiment>/<usage>.csv`, avec l'horizon `actuel` ou `2040`, un fichier par
usage (eau chaude, usages spécifiques, chauffage Joule, PAC, climatisation, recharge des véhicules) et une colonne
par année climatique. On prend ici l'année 2023.

In [ ]:
# consommation du bâtiment (kW)
df = pd.DataFrame(index=PV.index)
df["conso"] = ...

# Construisez la production solaire (kW)
P_pv = ...
df["pv"] = ...

for debut, fin, nom in [("2023-06-12", "2023-06-18", "été"), ("2023-01-16", "2023-01-22", "hiver")]:
    fig, ax = plt.subplots(figsize=(18, 3.5))
    ax.plot(df.loc[debut:fin, "conso"], color="black", lw=1, label="consommation")
    ax.fill_between(df.loc[debut:fin].index, df.loc[debut:fin, "pv"], color="orange", alpha=0.5, label="production solaire")
    ax.set_title(f"{type_bat}, semaine d'{nom} (consommation 2023)")
    ax.set_ylabel("kW")
    ax.set_ylim(0, 500)
    ax.tick_params(axis="x", labelrotation=45, labelsize=7)
    ax.legend(fontsize=7)
    fig.tight_layout()
    plt.show()

## 3. Autoconsommation et autoproduction

- **Taux d'autoconsommation** : part de la production solaire consommée sur place.
- **Taux d'autoproduction** : part de la consommation couverte par le solaire produit sur place.

Sans batterie, ce qui n'est pas consommé à l'heure où il est produit est injecté sur le réseau.

In [ ]:
# Calculez l'autoconsommation et l'autoproduction

taux_autoconsommation = ...
taux_autoproduction = ...

print(f"{P_pv} kWc")
print(f"Taux d'autoconsommation : {taux_autoconsommation:.0%}")
print(f"Taux d'autoproduction : {taux_autoproduction:.0%}")

## 4. Construire la courbe de prix

On fait tourner un modèle niveau européen en 2040, on en extrait les prix et on les utilise tels quels.

In [ ]:
PRIX = pd.read_csv("data/interne/prices_2040.csv", index_col="date_heure", sep=";", parse_dates=True, dayfirst=True)

# Afficher moyenne
print(PRIX.mean().round(2))

fig, axes = plt.subplots(1, 2, figsize=(10, 4))
axes[0].plot(PRIX["2040_FR"].groupby(PRIX.index.hour).mean(), label="2040_FR")
axes[1].plot(np.sort(PRIX["2040_FR"].values)[::-1], label="2040_FR")
axes[0].set_title("profil journalier moyen")
axes[0].set_xlabel("heure légale")
axes[0].set_ylabel("EUR/MWh")
axes[0].legend()
axes[1].set_title("monotone des prix")
axes[1].set_xlabel("heures classées")
axes[0].set_ylim(0, 60)
axes[1].set_ylim(0, 200)
for ax in axes:
    ax.grid(alpha=0.3)
fig.tight_layout()
plt.show()

Construire deux courbes de prix : plat et HP/HC, de manière à ce que la moyenne annuelle reste la même.

In [ ]:
# Prix spot du marché (EUR/MWh)
SPOT = PRIX["2040_FR"]
print(f"Prix spot moyen : {SPOT.mean():.1f} EUR/MWh")

# Semaine de mars, pour les graphiques
annee = PRIX.index.year[0]
debut, fin = f"{annee}-03-06", f"{annee}-03-12"

# Heures creuses (heure légale) : les mêmes pour le TURPE et pour le tarif HP/HC
HEURES_CREUSES = [23, 0, 1, 2, 3, 4, 5, 6]
creuse = PRIX.index.hour.isin(HEURES_CREUSES)

conso = df["conso"]
prod = df["pv"]

# Puissance max soutirée au réseau (= demande nette)
soutirage = ...
pointe = soutirage.max()
print(f"Puissance max soutirée : {pointe:.0f} kW")

In [ ]:
# Calcul du TURPE
# ...

# Saison TURPE de chaque heure
hiver = PRIX.index.month.isin([11, 12, 1, 2, 3])
turpe_energie = ... # np.where(hiver, TURPE_HIVER, TURPE_ETE)

# Calculez la part fixe du TURPE (EUR/an), puis la part fixe et la CTA ramenées à chaque MWh soutiré (EUR/MWh)
turpe_fixe = ...       # gestion + comptage + part puissance sur la pointe
fixe_par_mwh = ...     # (part fixe + CTA) / MWh soutirés sur l'année

# Construisez le prix spot complet HTC (EUR/MWh)
SPOT_COMPLET = ...
print(f"Prix spot complet moyen : {SPOT_COMPLET.mean():.1f} EUR/MWh HTC")

fig, ax = plt.subplots(figsize=(12, 4))
ax.plot(SPOT.loc[debut:fin], label="Spot seul")
ax.plot(SPOT_COMPLET.loc[debut:fin], label="Spot + fourniture, TURPE et taxes")
ax.set_title(f"Du prix spot au prix complet, {type_bat} (semaine de mars)")
ax.set_ylabel("EUR/MWh")
ax.grid(alpha=0.3)
ax.legend()
ax.set_ylim(0, 220)
plt.show()

In [ ]:
# Les trois tarifs ont le même prix moyen sur l'année : celui du spot complet
prix_moyen = SPOT_COMPLET.mean()

# Tarif SPOT
TARIF_SPOT = SPOT_COMPLET

# Construisez le tarif PLAT : le même prix à toutes les heures
TARIF_PLAT = ...

# Construisez le tarif HP/HC : prix_hc en heures creuses, RAPPORT_HP_HC × prix_hc en heures pleines, avec prix_hc
# choisi pour que la moyenne annuelle soit prix_moyen (creuse.mean() donne la part des heures creuses)
RAPPORT_HP_HC = 1.3   # prix HP / prix HC 
prix_hc = ...
TARIF_HPHC = ...

print(f"Prix moyens (EUR/MWh HTC) : spot {TARIF_SPOT.mean():.1f}, plat {TARIF_PLAT.mean():.1f}, HP/HC {TARIF_HPHC.mean():.1f}")
print(f"Tarif HP/HC : {prix_hc:.1f} EUR/MWh en HC, {RAPPORT_HP_HC * prix_hc:.1f} EUR/MWh en HP")

fig, ax = plt.subplots(figsize=(12, 4))
ax.plot(TARIF_SPOT.loc[debut:fin], label="Spot")
ax.plot(TARIF_PLAT.loc[debut:fin], label="Plat")
ax.step(TARIF_HPHC.loc[debut:fin].index, TARIF_HPHC.loc[debut:fin], where="post", label="HP/HC")
ax.set_title(f"Tarifs HTC spot, plat et HP/HC, {type_bat} (semaine de mars)")
ax.set_ylabel("EUR/MWh")
ax.grid(alpha=0.3)
ax.legend()
ax.set_ylim(0, 210)
plt.show()

In [ ]:
# Surplus solaire injecté sur le réseau (kW) : ce qui n'est pas consommé à l'heure où il est produit
surplus = ...
soutirage

print(f"Consommation : {conso.sum() / 1000:.0f} MWh")
print(f"Production solaire : {prod.sum() / 1000:.0f} MWh, dont {surplus.sum() / 1000:.0f} MWh injectés")
print(f"Soutirage au réseau : {soutirage.sum() / 1000:.0f} MWh")

# Prix de vente du surplus (EUR/MWh) : spot moins une part du fournisseur
PART_FOURNISSEUR = 5   # EUR/MWh, HYPOTHÈSE PÉDAGOGIQUE
prix_vente = pd.Series(SPOT - PART_FOURNISSEUR, index=PRIX.index)

# Calculez la revente du surplus (EUR/an)
revente = ...

# Calculez l'achat de l'électricité soutirée (EUR/an) avec chaque tarif : les tarifs contiennent déjà fourniture,
# TURPE et taxes
achat_spot = ...
achat_plat = ...
achat_hphc = ...

FACTURE = pd.DataFrame({"achat": [achat_spot, achat_plat, achat_hphc], "revente": revente}, index=["spot", "plat", "HP/HC"])
FACTURE["total"] = FACTURE["achat"] - FACTURE["revente"]
print(FACTURE.round(0))

fig, ax = plt.subplots(figsize=(6, 4))
ax.bar(FACTURE.index, FACTURE["achat"] / 1000, label="achat")
ax.bar(FACTURE.index, -FACTURE["revente"] / 1000, label="revente du surplus")
ax.scatter(FACTURE.index, FACTURE["total"] / 1000, color="black", zorder=3, label="total")
ax.axhline(0, color="black", lw=0.8)
ax.set_title(f"Facture annuelle, {type_bat}")
ax.set_ylabel("kEUR/an")
ax.grid(alpha=0.3, axis="y")
ax.legend()
plt.show()

In [ ]:
# Afficher pour chaque type de tarif la décomposition de la facture : énergie, taxe, réseau, injection


## 5. Projection 2040 : quel niveau de solaire pour les bureaux ?

En 2040, la consommation des bureaux est celle projetée en séance 1 (PAC, rénovation, flotte plus grande).
LPI hésite entre trois niveaux de solaire, ce sont les **décisions** :

- **aucun** : pas de solaire (référence : la loi APER impose pourtant des ombrières sur ce parking) ;
- **obligations** : la moitié du parking en ombrières et la moitié de la toiture, avec le parking agrandi pour la flotte 2040 ;
- **maximum** : tout le parking et toute la toiture, (toute la surface est utilisable).

Modélisation des trois années climatiques (2008, 2009, 2012).

In [ ]:
ANNEES = ["2008", "2009", "2012"]   # années climatiques (scénarios)

# Consommation 2040 des bureaux (kW) : somme des usages, une colonne par année climatique
CONSO_2040 = ...

# Parking 2040
SURFACE_PARKING = ...

# Construisez la puissance installée (kWc) de chaque niveau de solaire (décision)
KWC = pd.DataFrame({
    "ombriere": [0, ..., ...],
    "toiture": [0, ..., ...],
}, index=["aucun", "obligations", "maximum"])
KWC["total"] = KWC["ombriere"] + KWC["toiture"]
KWC.round(0)

In [ ]:
# Calculez l'investissement (EUR), puis l'annuité (EUR/an) pour chaque scénario d'installation de PV

KWC["investissement"] = ...
KWC["annuite"] = ...
KWC.round(0)

In [ ]:
# Tarif d'achat retenu pour 2040 (TARIF_SPOT, TARIF_PLAT ou TARIF_HPHC), tel que calculé plus haut :
# fourniture, TURPE (parts fixes ramenées au MWh) et taxes compris
TARIF_2040 = ...

# Pour chaque niveau de solaire et chaque année climatique : production, soutirage et surplus (kW), puis achat

lignes = []
for niveau in KWC.index:
    for annee in ANNEES:
        prod_2040 = ...
        soutirage_2040 = ...
        surplus_2040 = ...

        achat = ...
        revente_2040 = ...

        lignes.append({"niveau": niveau, "annee_climatique": annee, "kWc": KWC.loc[niveau, "total"],
                       "investissement": KWC.loc[niveau, "investissement"], "annuite PV": KWC.loc[niveau, "annuite"],
                       "achat net": achat - revente_2040, "cout global": KWC.loc[niveau, "annuite"] + achat - revente_2040})
DETAIL = pd.DataFrame(lignes)
DETAIL.round(0)   # EUR et EUR/an HT

In [ ]:
# Probabilité de chaque année climatique (à choisir)
PROBA = pd.Series({"2008": ..., 
                   "2009": ..., 
                   "2012": ...
                   })

# Coût global (EUR/an) : une ligne par niveau de solaire, une colonne par année climatique
COUTS = DETAIL.pivot(index="niveau", columns="annee_climatique", values="cout global").loc[KWC.index, ANNEES]

# Calculez le regret : coût de la décision moins coût de la meilleure décision de la même année climatique
REGRET = ...

TABLEAU = COUTS.copy()
TABLEAU.insert(0, "kWc", KWC["total"])
# Complétez les trois critères de chaque décision : espérance du coût (probabilités PROBA), coût maximal, regret maximal
TABLEAU["espérance"] = ...
TABLEAU["coût max"] = ...
TABLEAU["regret max"] = ...

print("Espérance :", TABLEAU["espérance"].idxmin())
print("Minimax du coût :", TABLEAU["coût max"].idxmin())
print("Minimax du regret :", TABLEAU["regret max"].idxmin())

# En gras : la meilleure décision de chaque année climatique (optimale), puis de chaque critère
TABLEAU.style.highlight_min(axis=0, subset=TABLEAU.columns[1:], props="font-weight: bold").format("{:.0f}")

**Conseil à LPI** : quel niveau de solaire recommandez-vous pour les bureaux ? Le choix change-t-il selon
le critère (espérance, minimax du coût, minimax du regret), selon les probabilités ou selon le tarif retenu ?

In [ ]:
# Agrégation : tous les bâtiments de bureaux de LPI en 2040 (identiques, un compteur chacun), espérance sur les années climatiques
N_BUREAUX = ...
poids = PROBA / PROBA.sum()   # probabilités ramenées à une somme de 1

# économie sur la facture d'un bâtiment (EUR/an) : achat net sans solaire moins achat net avec solaire
achat_net = DETAIL.pivot(index="niveau", columns="annee_climatique", values="achat net").loc[KWC.index, ANNEES]
achat_moyen = (achat_net * poids).sum(axis=1)
economie = ...

TOTAL = pd.DataFrame(index=KWC.index)
TOTAL["MWc installés"] = N_BUREAUX * KWC["total"] / 1000
TOTAL["investissement (MEUR)"] = N_BUREAUX * KWC["investissement"] / 1e6
TOTAL["annuité PV (kEUR/an)"] = N_BUREAUX * KWC["annuite"] / 1000
TOTAL["économie sur la facture (kEUR/an)"] = N_BUREAUX * economie / 1000
# Complétez le gain net (économie moins annuité) et le temps de retour simple, sans actualisation :
# investissement / (économie annuelle moins exploitation)
TOTAL["gain net (kEUR/an)"] = ...
TOTAL["temps de retour (ans)"] = ...

print(f"Ensemble des {N_BUREAUX} bâtiments de bureaux en 2040 (EUR HT)")
TOTAL.round(1)

In [ ]:
# Impact sur le réseau de l'ensemble des bureaux, mêmes colonnes qu'à la fin de la séance 1, sans puis avec solaire :
# énergie et pointe nettes de la production solaire ; émissions du soutirage (contenu CO2 moyen horaire éCO2mix 2023,
# solaire compté à zéro)
CO2 = pd.read_csv("data/interne/courbe_co2.csv", index_col="date_heure", parse_dates=True)["co2_g_par_kwh"]
N_BUREAUX_ACTUEL = ...

CONSO_ACTUEL = ...

# scénarios : (nombre de bâtiments, consommation d'un bâtiment, kWc par bâtiment) ; actuel = installation de la partie 2
SCENARIOS = {"actuel": (N_BUREAUX_ACTUEL, CONSO_ACTUEL, P_pv)}
for niveau in KWC.index:
    SCENARIOS[f"2040 {niveau}"] = (N_BUREAUX, CONSO_2040, KWC.loc[niveau, "total"])

impact = {}
for scenario, (n, conso_scenario, kwc) in SCENARIOS.items():
    for annee in ANNEES:
        brut = n * conso_scenario[annee]          # kW, sans solaire
        net = ...                                 # kW, avec solaire (négatif quand le solaire dépasse la consommation)
        # Complétez les émissions avec solaire : seul le soutirage (la partie positive de net) émet
        impact[(scenario, annee)] = {"energie_sans_pv_gwh": brut.sum() / 1e6, "energie_gwh": net.sum() / 1e6,
                                     "pointe_sans_pv_mw": brut.max() / 1000, "pointe_mw": net.max() / 1000,
                                     "emissions_sans_pv_tco2": (brut * CO2).sum() / 1e6, "emissions_tco2": ...}
RESTITUTION = pd.DataFrame(impact).T   # sans solaire : les valeurs de la fin de la séance 1
RESTITUTION.round(2)

In [ ]:
ordre = ["actuel", "2040 aucun", "2040 obligations", "2040 maximum"]
couleurs = ["tab:blue", "tab:orange", "tab:green", "tab:red"]
largeur = 0.2
x = np.arange(len(ANNEES))

fig, axes = plt.subplots(1, 3, figsize=(15, 3.8))
for ax, (col, titre) in zip(axes, [("energie", "Énergie (GWh/an)"), ("pointe", "Pointe (MW)"),
                                   ("emissions", "Émissions, contenu moyen 2023 (tCO2/an)")]):
    unite = {"energie": "gwh", "pointe": "mw", "emissions": "tco2"}[col]
    for i, (scenario, couleur) in enumerate(zip(ordre, couleurs)):
        position = x + (i - 1.5) * largeur
        valeurs = RESTITUTION.loc[scenario].loc[ANNEES]
        ax.bar(position, valeurs[f"{col}_sans_pv_{unite}"], largeur, fill=False, edgecolor=couleur, linestyle="--")   # sans solaire
        ax.bar(position, valeurs[f"{col}_{unite}"], largeur, color=couleur, label=scenario)                           # avec solaire
    ax.set_xticks(x, ANNEES)
    ax.set_title(titre)
    ax.set_xlabel("année climatique")
    ax.grid(alpha=0.3)
axes[-1].legend(fontsize=8, bbox_to_anchor=(1.01, 1), loc="upper left")
fig.suptitle("Ensemble des bureaux de LPI : barre pleine avec solaire, contour en pointillés sans solaire")
fig.tight_layout()
plt.show()